# 06 - Time histories and HDF5 archive (Sinha, 2007)

This notebook is the **simulation-to-disk** step for the Sinha (2007) validation study: it turns ROSS time-domain runs into **archived, sensor-like traces** that later notebooks can load for bispectrum, trispectrum, and fault-discrimination analyses.

## What this step does

1. **Loads** the Sinha rotor from `sinha_rotor.toml` and picks a default lateral probe (see `sinha_tools`).
2. **Integrates** each fault case at selected shaft speeds using `simulate_case` (healthy, crack with pseudomodal reduction, misalignment variants). The output is the **vertical displacement** at the probe, sampled at `fs_sim = 2560` Hz.
3. **Discards** the first **5 s** of each record as transient, keeping a **steady-state** window for analysis (26 s total simulated when `FAST` is off, so **21 s** retained at 2560 Hz before acquisition; 2 s total when `FAST=1` for quick checks).
4. **Applies an acquisition chain** meant to mimic measured data: **1 kHz low-pass**, **10× block decimation** (2560 Hz → 256 Hz effective), then **additive white Gaussian noise** at **40 dB SNR**. That yields a single-channel series and `fs_out` stored with the file.
5. **Writes** one HDF5 file per `(case, rpm)` under `study_docs/sinha_validation/hdf5/`, with arrays `t`, `y`, sampling rate, and JSON `meta` (case name, RPM, input/output rates, probe node). Downstream notebooks read these files instead of re-running long simulations.

Together, this isolates **what we assume about the instrument and record length** from **what the rotor model does**, so higher-order spectral notebooks only depend on a stable file format.

## Assumptions ledger

- **Simulation sample rate:** `fs_sim = 2560` Hz, so `dt = 1/2560` s. Sinha (2007) does not state the finite-element integration step used in the original work; here the rate is chosen so that, after **10× decimation**, the **256 Hz** band matches a convenient analysis grid for overlap–segment processing (see later notebooks).
- **Record length:** With `FAST=0`, **26 s** are simulated and **5 s** are discarded, leaving **21 s** of steady data at **2560 Hz** on `y_ss`. That length supports **50 segments** with **50% overlap** and **nfft = 2048** if those settings are applied at the **pre-decimation** rate. The archived HDF5 series is the **post-acquisition** signal (e.g. ~21 s at **256 Hz** after decimation for the default parameters), shorter in sample count but matched to `fs_out` in `meta`.
- **Spectral settings (Hanning, nfft=2048):** Used in follow-on notebooks for STFT / HOS; they are **modelling choices**, not parameters quoted verbatim from Sinha (2007).
- **FAST mode:** Set environment variable `FAST=1` to shorten wall time (fewer simulated seconds, fewer modes); use only for smoke tests, not for publication-quality archives.

## References

Sinha, J. K. (2007). Higher order spectra for crack and misalignment identification in the shaft of a rotating machine. *Structural Health Monitoring*, 6(4), 325-334.


In [1]:
import os
import sys
from pathlib import Path

import numpy as np
import ross as rs

if "." not in sys.path:
    sys.path.insert(0, ".")
import sinha_tools as st

FAST = os.environ.get("FAST", "0") == "1"
OUT = Path("study_docs/sinha_validation/hdf5")
OUT.mkdir(parents=True, exist_ok=True)

rotor = st.load_sinha_rotor("sinha_rotor.toml")
probe = st.default_probe(rotor)
fs_sim = 2560.0
dt = 1.0 / fs_sim
n_total = int(26.0 * fs_sim) if not FAST else int(2.0 * fs_sim)
t_disc = int(5.0 * fs_sim)
t = np.arange(0, n_total, dtype=float) * dt
print("samples", len(t), "FAST", FAST)


samples 66560 FAST False


In [2]:
Q_ = rs.Q_
num_modes = 16 if FAST else 24

cases = []
for rpm in (750, 900):
    cases.append(("healthy", rpm))
for rpm in (650, 750):
    cases.append(("crack", rpm))
for rpm in range(300, 901, 150):
    cases.append(("misalignment", rpm))
for rpm in (750, 900):
    cases.append(("misalignment_pedestal", rpm))

for case, rpm in cases:
    speed = rpm * 2 * np.pi / 60.0
    print(case, rpm)
    res = st.simulate_case(rotor, case, speed, t, num_modes=num_modes)
    y = st.probe_displacement_y_from_rotor(rotor, res, probe)
    y_ss = y[t_disc:]
    y_acq, fs_out = st.acquisition_chain(y_ss, fs_sim, lp_hz=1000.0, decimate_factor=10, snr_db=40.0, seed=42)
    meta = {"case": case, "rpm": rpm, "fs_in_hz": fs_sim, "fs_out_hz": fs_out, "probe_node": int(probe.node)}
    fname = OUT / f"{case}_{rpm}rpm.h5"
    t_out = np.arange(len(y_acq), dtype=float) / fs_out
    st.save_timeseries_hdf5(fname, fs=fs_out, t=t_out, y=y_acq, meta=meta)
    print("  wrote", fname, "len", len(y_acq))


healthy 750
  wrote study_docs/sinha_validation/hdf5/healthy_750rpm.h5 len 5376
healthy 900
  wrote study_docs/sinha_validation/hdf5/healthy_900rpm.h5 len 5376
crack 650
Running with model reduction: pseudomodal
  wrote study_docs/sinha_validation/hdf5/crack_650rpm.h5 len 5376
crack 750
Running with model reduction: pseudomodal
  wrote study_docs/sinha_validation/hdf5/crack_750rpm.h5 len 5376
misalignment 300
  wrote study_docs/sinha_validation/hdf5/misalignment_300rpm.h5 len 5376
misalignment 450
  wrote study_docs/sinha_validation/hdf5/misalignment_450rpm.h5 len 5376
misalignment 600
  wrote study_docs/sinha_validation/hdf5/misalignment_600rpm.h5 len 5376
misalignment 750
  wrote study_docs/sinha_validation/hdf5/misalignment_750rpm.h5 len 5376
misalignment 900
  wrote study_docs/sinha_validation/hdf5/misalignment_900rpm.h5 len 5376
misalignment_pedestal 750
  wrote study_docs/sinha_validation/hdf5/misalignment_pedestal_750rpm.h5 len 5376
misalignment_pedestal 900
  wrote study_docs/s